<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 2</b>

## Unit 4: LangChain Fundamentals — LCEL and Chains

**CV Raman Global University, Bhubaneswar**
*AI Center of Excellence*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Use Chat Models** — work with `ChatOpenAI` and LangChain message types (`HumanMessage`, `SystemMessage`, `AIMessage`)
2. **Create Prompt Templates** — build reusable prompts with `ChatPromptTemplate` and template variables
3. **Understand LCEL** — use the pipe `|` syntax to chain components together
4. **Build Chains** — compose chains with `prompt | llm | parser` and add custom post-processing with `RunnableLambda`

## 1. Environment Setup

In [ ]:
!pip install -q langchain langchain-openai pydantic

In [ ]:
import os
from getpass import getpass

api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key
MODEL = "gpt-4o-mini"

---

## 2. Chat Models

LangChain wraps LLM providers behind a unified interface. `ChatOpenAI` is the wrapper for OpenAI's chat models.

LangChain uses three **message types** to represent a conversation:

| Message Type | Role | Purpose |
|---|---|---|
| `SystemMessage` | system | Sets the assistant's behavior and personality |
| `HumanMessage` | user | The user's input |
| `AIMessage` | assistant | The model's response |

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

llm = ChatOpenAI(model=MODEL)

### Invoking with a Simple String

The simplest way to call a chat model — pass a string directly:

In [ ]:
response = llm.invoke("What is the capital of France?")
print(response.content)

### Invoking with Message Objects

For more control, pass a list of message objects. This lets you set a system prompt and build multi-turn conversations:

In [ ]:
messages = [
    SystemMessage(content="You are a helpful science tutor. Keep answers to one sentence."),
    HumanMessage(content="What causes rain?"),
]

response = llm.invoke(messages)
print(response.content)
print(f"\nType: {type(response)}")

The response is an `AIMessage` object. You can use it to continue the conversation:

In [ ]:
# Continue the conversation by appending the AI response and a follow-up
messages.append(response)
messages.append(HumanMessage(content="What about snow?"))

response2 = llm.invoke(messages)
print(response2.content)

### Model Parameters

You can control the model's behavior with parameters like `temperature`:

In [ ]:
# Low temperature = more deterministic
precise_llm = ChatOpenAI(model=MODEL, temperature=0)
response = precise_llm.invoke("What is 2 + 2?")
print(f"Precise: {response.content}")

# High temperature = more creative
creative_llm = ChatOpenAI(model=MODEL, temperature=1.0)
response = creative_llm.invoke("Write a one-line poem about Python programming.")
print(f"Creative: {response.content}")

> **Key takeaway:** `ChatOpenAI` provides a unified interface to OpenAI models. You interact with it using message objects (`SystemMessage`, `HumanMessage`, `AIMessage`) and get back `AIMessage` responses.

---

## 3. Prompt Templates

Hardcoding prompts is brittle. **Prompt templates** let you define reusable prompts with variables that get filled in at runtime.

`ChatPromptTemplate` creates a template that produces a list of messages:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

### Basic Template

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that explains {topic} concepts simply."),
    ("human", "{question}"),
])

# Inspect the template variables
print("Input variables:", prompt.input_variables)

In [ ]:
# Format the template with values
formatted = prompt.invoke({"topic": "physics", "question": "What is gravity?"})
print(formatted.messages)

In [ ]:
# Pass the formatted prompt to the LLM
response = llm.invoke(formatted)
print(response.content)

### Why Templates Matter

Templates separate **prompt logic** from **prompt data**. The same template works for any topic:

In [ ]:
for topic, question in [("biology", "What is DNA?"), ("math", "What is a prime number?")]:
    response = llm.invoke(prompt.invoke({"topic": topic, "question": question}))
    print(f"[{topic.upper()}] {response.content}\n")

> **Key takeaway:** `ChatPromptTemplate` creates reusable prompts with `{variables}`. Use `from_messages()` to define system and human messages with placeholders.

---

## 4. LCEL (LangChain Expression Language)

So far we've been manually passing outputs between components:

```python
formatted = prompt.invoke(data)
response = llm.invoke(formatted)
```

**LCEL** lets you chain these with the pipe `|` operator into a single **Runnable** chain:

```python
chain = prompt | llm | parser
result = chain.invoke(data)
```

Each component's output becomes the next component's input — just like Unix pipes.

### Your First Chain

`StrOutputParser` extracts the text content from an `AIMessage` as a plain string — it's the most common way to get usable text output from a chain.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep answers to one sentence."),
    ("human", "{question}"),
])

# Chain: prompt -> model -> parser
chain = prompt | llm | StrOutputParser()

# invoke() runs the entire chain
result = chain.invoke({"question": "What is photosynthesis?"})
print(result)
print(f"\nType: {type(result)}")

### How the Pipe Works

```
{"question": "What is photosynthesis?"}
    │
    ▼
┌─────────────────┐
│  Prompt Template │  → produces ChatPromptValue (list of messages)
└─────────────────┘
    │
    ▼
┌─────────────────┐
│    ChatOpenAI    │  → produces AIMessage
└─────────────────┘
    │
    ▼
┌─────────────────┐
│ StrOutputParser  │  → produces str
└─────────────────┘
    │
    ▼
"Photosynthesis is..."
```

Every component in LangChain implements the **Runnable** interface with `.invoke()`. The `|` operator connects them so that `a | b` means "call `a.invoke()`, then pass the result to `b.invoke()`".

### Pydantic Output Parser

`StrOutputParser` gives you plain text. But what if you need **structured data** — like a Python object with typed fields?

`PydanticOutputParser` lets you define the expected output shape using a Pydantic model. It:
1. Generates **format instructions** that tell the LLM exactly how to structure its response (as JSON)
2. **Parses** the LLM's response into a validated Pydantic object

First, let's define a Pydantic model and see the format instructions:

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Define the expected output structure
class CountryInfo(BaseModel):
    name: str = Field(description="Name of the country")
    capital: str = Field(description="Capital city")
    population: int = Field(description="Approximate population")
    fun_fact: str = Field(description="An interesting fact about the country")

# Create the parser
pydantic_parser = PydanticOutputParser(pydantic_object=CountryInfo)

# See what format instructions the parser generates for the LLM
print(pydantic_parser.get_format_instructions())

Now let's use the parser in an LCEL chain. The key trick: inject the format instructions into the prompt using `partial_variables`, so the LLM knows what JSON structure to produce:

In [ ]:
# Build a prompt that includes format instructions
country_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Always respond in the exact format requested."),
    ("human", "Tell me about {country}.\n\n{format_instructions}"),
])

# Chain: prompt -> model -> pydantic parser
country_chain = country_prompt | llm | pydantic_parser

# Run the chain
result = country_chain.invoke({
    "country": "Japan",
    "format_instructions": pydantic_parser.get_format_instructions(),
})

# The result is a validated Pydantic object, not a string!
print(f"Type: {type(result)}\n")
print(f"Name:       {result.name}")
print(f"Capital:    {result.capital}")
print(f"Population: {result.population:,}")
print(f"Fun fact:   {result.fun_fact}")

### RunnableLambda

LCEL provides `RunnableLambda` to wrap any Python function as a Runnable, so you can plug custom logic into a chain:

In [ ]:
from langchain_core.runnables import RunnableLambda

# RunnableLambda: wraps any function as a Runnable
def word_count(text: str) -> str:
    count = len(text.split())
    return f"{text} (word count: {count})"

word_counter = RunnableLambda(word_count)
print(word_counter.invoke("Hello world from LangChain"))

In [ ]:
# Use RunnableLambda as a post-processing step in a chain
def to_uppercase(text: str) -> str:
    return text.upper()

chain = prompt | llm | StrOutputParser() | RunnableLambda(to_uppercase)

result = chain.invoke({"question": "What is the largest planet?"})
print(result)

> **Key takeaway:** LCEL's pipe `|` operator chains components together: `prompt | model | parser`. Every LangChain component is a **Runnable** with `.invoke()`. Use `RunnableLambda` to wrap custom Python functions and plug them into chains.

---

## 5. Key Takeaways

### Concept Map

```
ChatOpenAI(model="gpt-4o-mini")          ─── the LLM wrapper
  │
  ├── .invoke(messages)                   ─── call the model
  │     └── returns AIMessage
  │
ChatPromptTemplate                        ─── reusable prompt with {variables}
  │
  ├── .from_messages([(role, template)])   ─── define system + human messages
  ├── .invoke({vars})                     ─── fill in the variables
  │
LCEL (pipe operator)                      ─── chain components together
  │
  ├── prompt | llm | parser               ─── basic chain pattern
  ├── StrOutputParser()                   ─── AIMessage → str
  ├── PydanticOutputParser(model)         ─── AIMessage → Pydantic object
  └── RunnableLambda(fn)                  ─── wrap any function as Runnable
```

### Quick Reference

| Concept | Code | Purpose |
|---|---|---|
| **Chat Model** | `ChatOpenAI(model=MODEL)` | Wrapper for OpenAI chat models |
| **Messages** | `HumanMessage`, `SystemMessage`, `AIMessage` | Represent conversation turns |
| **Prompt Template** | `ChatPromptTemplate.from_messages([...])` | Reusable prompts with variables |
| **String Parser** | `StrOutputParser()` | AIMessage → plain string |
| **Pydantic Parser** | `PydanticOutputParser(pydantic_object=Model)` | AIMessage → validated Pydantic object |
| **LCEL Pipe** | `prompt \| llm \| parser` | Chain components together |
| **Lambda** | `RunnableLambda(fn)` | Wrap any function as Runnable |

---

## 6. Exercises

### Exercise 1: Translation Chain
Build an LCEL chain that takes `{text}` and `{language}` as input and returns the translated text as a string. Use `ChatPromptTemplate` with a system message and `StrOutputParser`. Test it with at least two different languages.

### Exercise 2: Post-Processing with RunnableLambda
Build a chain that answers a `{question}` in one sentence, then use `RunnableLambda` to add a character count and word count to the output (e.g., `"Answer text (chars: 42, words: 8)"`).

---

**What's Next?** In the next notebook, we'll use these LangChain concepts to build **LangGraph** applications — stateful agents with tools, conditional routing, and memory.

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 2
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---